# Product Recommendation Agent - Class Notebook

This notebook builds a simple agentic product recommendation system step by step. You will learn:

1. **Pydantic** - how to define structured data schemas and validation.
2. **Web Search Tools** - using DuckDuckGo to fetch live product information.
3. **LangChain + OpenAI** - setting up an LLM and structured output chains.
4. **Multi-agent pipeline** - discovery, specs research, price check, reviews, and final verdict.

Run the cells in order; later cells reuse variables defined earlier.


# Pydantic Introduction

## Section 1: Pydantic Data Validation Basics

Pydantic is a Python library that turns dictionaries/classes into validated data models. In LLM apps it is used to force the model to return structured JSON with the right fields and types.


Install the `pydantic` package so we can define data models that validate field types automatically.


In [9]:
!pip install pydantic

Defaulting to user installation because normal site-packages is not writeable


Import `BaseModel` (base class for schemas) and `Field` (adds descriptions and defaults).


In [8]:
from pydantic import BaseModel, Field

Define a simple `transcript` model. Each annotation (`str`) tells Pydantic the required type. Missing or wrong-typed values raise a validation error.


In [2]:
class transcript(BaseModel):
    headline : str
    content : str
    conclusion : str
    

Create a valid `transcript` instance. Pydantic accepts keyword arguments and stores them as attributes.


In [4]:
tr = transcript(headline = "This is introduction on pydantic", content = "Pydantic is a data validation framework", conclusion = "Use pydantic for data validation in LLM Application")

Access the `headline` attribute to confirm the object was created correctly.


In [6]:
tr.headline

'This is introduction on pydantic'

Access the `content` attribute.


In [7]:
tr.content

'Pydantic is a data validation framework'

Access the `conclusion` attribute.


In [8]:
tr.conclusion

'Use pydantic for data validation in LLM Application'

Try creating an instance with `headline=None`. Because `headline` is typed `str` (not `Optional[str]`), Pydantic rejects `None`. Expect a validation error.


In [10]:
tr = transcript(headline = None, content = "Pydantic is a data validation framework", conclusion = "Use pydantic for data validation in LLM Application")

ValidationError: 1 validation error for transcript
headline
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

Expected result: Pydantic raises a `ValidationError` because `None` is not a valid `str`.


Redefine `transcript` with `Field(...)`. `Field` lets us add human-readable descriptions and default values (e.g., `views` defaults to `0`).


In [12]:
class transcript(BaseModel):
    headline : str = Field(description="5-10 word headline for the transcript")
    content : str
    conclusion : str
    views : int = Field(default=0,description = "The number of views in the youtube video")

Create an instance of the improved model. Only `headline`, `content`, and `conclusion` are required; `views` will default to `0`.


In [13]:
transcript(headline = "ABC", content = "XYZ", conclusion = "vfhd")

transcript(headline='ABC', content='XYZ', conclusion='vfhd', views=0)

## Section 2: Installing & Using DuckDuckGo Search

The agent needs live web data. `DuckDuckGoSearchRun` is a LangChain tool that performs anonymous web searches.


Install the LangChain ecosystem, DuckDuckGo search wrapper, and `pydantic`/`ddgs` dependencies needed by the rest of the notebook.


In [14]:
!pip install langchain langchain-openai langchain-community duckduckgo-search pydantic ddgs

Defaulting to user installation because normal site-packages is not writeable


Import the DuckDuckGo search tool from LangChain's community package.


In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

C:\Users\DELL\AppData\Local\Temp\ipykernel_9112\1490969448.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


Create a `DuckDuckGoSearchRun` instance and search for laptop brands. The result is a string containing snippets from the web.


In [4]:
search = DuckDuckGoSearchRun()

search.run("I want the latest laptop brands")

'... the latest laptop brands ranking 2018 served for you all. MyWiseCart provides you with ... As such it has fallen in the latest laptop brand rankings. A: The best laptop brand 2021 is typically a top-of-the range, high performance computer with all the required components. Finding the right laptop brand is becoming strenuous every day, as new laptop brands are investing in the tech-markets. ... it comes to laptops, Lenovo is one of the top and ... Being a “unique” brand, Apple laptops have their own pre-installed OS called the Mac OS. MSI is a well-established brand in the gaming laptop market, known for producing high-performance machines under the GS, GT, and GE series.'

Expected result: a long string of web snippets related to current laptop brands in India.


Run a second search to show the tool works for any query. This returns live search snippets.


In [5]:
search.run("IPL 2025 winners")

'3 days ago - In this stage, the top two teams compete with each other (in a match titled "Qualifier 1"), as does the remaining two teams (in a match titled "Eliminator"). While the winner of Qualifier 1 would directly qualify for the final match, the losing team had another chance to qualify for the final match by competing against the winning team of the Eliminator (in a match titled "Qualifier 2"). The winner of this subsequent Qualifier 2 match advanced to the final match. The IPL Governing Council decided to keep the number of matches to 74 for this edition as it was in the previous three seasons to help the cricketers balance their workload; however, it will be increased to 84 from 2026 onwards with the return of the double round-robin format. 3 days ago - It was played between Royal Challengers Bengaluru and Punjab Kings. Punjab won the toss and elected to field first; Bengaluru scored 190/9 in their innings. In the second innings, Punjab scored 184/7. Bengaluru won the match by

Expected result: web snippets about IPL 2025 winners.


## Section 3: Project Setup & LLM Initialization

Now we set up the environment, load API keys from a `.env` file, and create an LLM instance.


Import all libraries used in the recommendation agent and call `load_dotenv(override=True)` so API keys stored in `.env` are available. `ChatOpenAI` is initialized with `gpt-4o-mini`.


In [6]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from datetime import date
from typing  import List, Optional


load_dotenv(override=True)

True

Print the current working directory so you know where `.env` and saved files are located.


In [2]:
pwd

'E:\\BIA\\BIA_GenAI_May_26'

Create the LLM object. `temperature=0` makes outputs more deterministic, which is good for structured data extraction.


In [7]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0)

Send a quick greeting to verify the LLM and API key are working. Expect a short assistant response.


In [9]:
llm.invoke("Hi there")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f8d40babb7', 'id': 'chatcmpl-DnfKOjoNpXoHLsWIc8GYajkJx6Rq2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9bbc-dc8b-7341-8e90-e2075e48a96d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Expected result: an `AIMessage` from `gpt-4o-mini` confirming the LLM connection works.


Show today's date; used later to make search queries current.


In [9]:
date.today()

datetime.date(2026, 6, 6)

## Section 4: Product Discovery Tool

The first agent tool searches the web for product candidates in a category.


Define `discover_products_tool` decorated with `@tool`. It builds a query like 'best laptops current models overview articles as of ... in India' and returns search results.


In [11]:
@tool
def discover_products_tool(category_query:str) -> str:
    """Searches the web to find lists, roundups, or recommendations for products within a specific category or budget"""
    
    search = DuckDuckGoSearchRun()
    today = date.today()
    country = "India"
    try:
        return search.run(f"best {category_query} current models overview articles as of {today} in {country}")
    except Exception as  e:
    
        return f"Error discovering the products : {str(e)}"

Invoke the tool with `category_query='laptops'`. This returns raw web text that will be parsed by the LLM.


In [12]:
discover_products_tool.invoke(input = {"category_query":"laptops"})

'In 2026, picking a laptop isn\'t a simple choice between "cheap" and "expensive" anymore. The market is overflowing with everything from scrappy productivity hubs to absolute desktop-crushing beasts. The hard part isn\'t finding a "good" machine, but finding the one that won\'t make your wallet cry. That\'s why we built this guide. Our Testing Process Looking for a new laptop in 2026? Here are the best consumer laptops in India, from affordable everyday picks to high-end productivity machines. Looking for recently launched Laptops in India? Find a list of latest Laptops in India from popular brands like HP, Dell, Lenovo, Asus, Apple at giznext.com. Looking for the best Performance laptops with? Discover top-rated options that deliver exceptional performance, battery life, and display quality tailored to your needs. Here at PCMag, we\'ve tested thousands of laptops since our lab\'s founding more than 40 years ago. Our analysts and editors have more than a collective century of experien

Expected result: raw search text containing laptop model names and recommendations.


Inspect the tool object. LangChain's `@tool` decorator creates a `StructuredTool` with name, description, and argument schema.


In [13]:
discover_products_tool

StructuredTool(name='discover_products_tool', description='Searches the web to find lists, roundups, or recommendations for products within a specific category or budget', args_schema=<class 'langchain_core.utils.pydantic.discover_products_tool'>, func=<function discover_products_tool at 0x000001A5FEEE5FE0>)

## Section 5: Structured Product Discovery Agent

Raw search text is hard to use. We define a Pydantic schema and an LLM chain that extracts exact product names.


Define the `DiscoveredProducts` schema. It confirms the category and lists 2-3 exact model names found in the search results.


In [14]:
class DiscoveredProducts(BaseModel):
    category_confirmed: str = Field(description = "The generic category of the products being searched")
    products_name : List[str] = Field(description="A list of top 2-3 specific, exact product model names found matching the user query.")

Create a `ChatPromptTemplate` that tells the LLM to act as a product-discovery agent and output only specific model names.


In [15]:
discovery_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a product discovery agent. Analyze the broad user request and search data to identify 2 to 3 exact product models that fit the criteria. Output only their specific names."),
    ("human", "User criteria: {user_query}. Search findings: {search_context}")
])

Display the prompt object to see how messages and variables are structured.


In [16]:
discovery_prompt

ChatPromptTemplate(input_variables=['search_context', 'user_query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a product discovery agent. Analyze the broad user request and search data to identify 2 to 3 exact product models that fit the criteria. Output only their specific names.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['search_context', 'user_query'], input_types={}, partial_variables={}, template='User criteria: {user_query}. Search findings: {search_context}'), additional_kwargs={})])

Build a chain: `discovery_prompt | llm.with_structured_output(DiscoveredProducts)`. The LLM will return an object matching the schema instead of free text.


In [17]:
structured_discoverer = discovery_prompt | llm.with_structured_output(DiscoveredProducts)

Set the user request that drives the rest of the demo.


In [18]:
user_query = "Top laptop models that I can buy"

Run the discovery tool against the user's query.


In [19]:
discovery_raw_data = discover_products_tool.invoke({"category_query": user_query})

Inspect the raw web results that were fetched.


In [20]:
discovery_raw_data

'In 2026, picking a laptop isn\'t a simple choice between "cheap" and "expensive" anymore. The market is overflowing with everything from scrappy productivity hubs to absolute desktop-crushing beasts. The hard part isn\'t finding a "good" machine, but finding the one that won\'t make your wallet cry. That\'s why we built this guide. Our Testing Process Our analysts and editors have more than a collective century of experience telling the good laptops from the great ones. We test more than 100 models every year to determine the best laptop overall. We enlisted a dozen testers to put 20 laptops through rigorous testing. These are the best laptops based on performance, battery life, display quality and more. Looking for the best Performance laptops with? Discover top-rated options that deliver exceptional performance, battery life, and display quality tailored to your needs. Here are our picks for the best laptops you can buy right now, whether you need an all-arounder for everyday use, a

Pass the raw search context and user query into the structured discovery chain. The LLM parses the text into `DiscoveredProducts`.


In [21]:
discovered_data = structured_discoverer.invoke({
        "user_query": user_query,
        "search_context": discovery_raw_data})

Expected result: a `DiscoveredProducts` object with `category_confirmed` and `products_name` populated by the LLM.


Display the parsed discovery object.


In [22]:
discovered_data

DiscoveredProducts(category_confirmed='Laptops', products_name=['Apple MacBook Pro 16-inch (2023)', 'Dell XPS 15 (2023)', 'Asus ROG Zephyrus G14 (2023)'])

Access the confirmed category field.


In [23]:
discovered_data.category_confirmed

'Laptops'

Access the list of discovered product names.


In [24]:
discovered_data.products_name

['Apple MacBook Pro 16-inch (2023)',
 'Dell XPS 15 (2023)',
 'Asus ROG Zephyrus G14 (2023)']

Inspect the `DiscoveredProducts` class itself to see the schema and field descriptions.


In [25]:
DiscoveredProducts

__main__.DiscoveredProducts

## Section 6: Technical Specifications Research Agent

Once we have product names, the next agent fetches official hardware/software specs.


Define `research_technical_specs`. It searches for technical specifications/datasheets for a given product model.


In [26]:
@tool
def research_technical_specs(product_name : str) -> str:
    """Searches the web for official manufacturer documentation and hardware speccifications for a specific product model"""

    search = DuckDuckGoSearchRun()
    try:
        return search.run(f"{product_name} officil technical specifications datasheet hardware specs")
    except Exception as e:
        return f"Error fetching  technical spec : {str(e)}"

Define the `TechnicalSpecs` schema to hold the official product name, key specs, and a short summary.


In [27]:
class TechnicalSpecs(BaseModel):
    product_name: str = Field(description="The exact official name of the product")
    key_specs: List[str] = Field(description="List of core technical specifications found")
    summary : str = Field(description="A brief 2-sentence summary of what this product is")

Create a researcher prompt and chain. The LLM extracts objective facts from web context.


In [28]:
researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert technical researcher. Use the tool context to extract objective hardware/software facts for the given product model. Filter out marketing fluff."),
    ("human", "Extract specs for product model: {product_to_research}. Context data: {tool_context}")

])
structured_researcher = researcher_prompt | llm.with_structured_output(TechnicalSpecs)


Loop over discovered products, fetch specs for each, and parse them with the structured researcher. `break` stops after the first product for brevity.


In [29]:
for product in discovered_data.products_name:
    print(f" -- processing : {product} --- ")

    print(f"Agent 1:Fetching technical specifications ... ")
    spec_raw_data = research_technical_specs.invoke({"product_name" : product})
    print(spec_raw_data)
    specs_output = structured_researcher.invoke({"product_to_research":product,"tool_context":spec_raw_data})
    print("-------specs Output -----")
    print(specs_output)
    break

 -- processing : Apple MacBook Pro 16-inch (2023) --- 
Agent 1:Fetching technical specifications ... 
15 Sept 2025 · MacBook Pro Technical Specifications · 18-core CPU with 6 super cores and 12 performance cores · 32-core GPU · Neural Accelerators · Hardware-accelerated ray ... 2 Oct 2025 · Testing conducted by Apple in January and February 2026 using preproduction 16-inch MacBook Pro systems with Apple M5 Pro, 18‑core CPU, 20‑core GPU, ... 16 Jun 2025 · From the Apple menu in the corner of your screen, choose About This Mac. The window that will open includes the model name and serial number. 25 Mar 2026 · Identify your MacBook Air model · Find the serial number printed on the underside of your Mac, near the regulatory markings. · The original packaging may also ... 14 Jan 2026 · Explore the Tata Punch specifications including engine details, dimension, transmission, technology, features & fuel tank capacity.
-------specs Output -----
product_name='Apple MacBook Pro 16-inch (2023)' ke

Expected result: printed raw specs and a parsed `TechnicalSpecs` object for the first product.


## Section 7: Budget Routing & Price Evaluation

Users may mention a budget. The router detects budget intent, and the price agent checks whether products fit.


Define `PriceFilterIntent`. The LLM reads the user query and decides whether a budget constraint exists and extracts `max_budget_inr`.


In [30]:
class PriceFilterIntent(BaseModel):
    has_budget_constraint: bool = Field(description = "True if the user specifies a maximum price or budget limit")
    max_budget_inr: Optional[float] = Field(description= "The maximum INR amount specified by the user, if any.")

Build the router chain. It takes a user query and returns a structured `PriceFilterIntent` object.


In [31]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the user query and determine if they are looking for a product under a specific price or budget constraint."),
    ("human", "{user_query}")
])


router_chain = router_prompt | llm.with_structured_output(PriceFilterIntent)


Display the current `user_query` value so we know what the router will analyze.


In [33]:
user_query

'Top laptop models that I can buy'

Invoke the router on the current query. Expect `has_budget_constraint=False` because the query does not mention a price limit.


In [32]:
router_chain.invoke(user_query)

PriceFilterIntent(has_budget_constraint=False, max_budget_inr=None)

Expected result: `has_budget_constraint=False` because the current query has no price limit.


Test the router with a budget query. Expect `has_budget_constraint=True` and a parsed INR amount.


In [34]:
router_chain.invoke("Get me the laptps within the budge of 1l")

PriceFilterIntent(has_budget_constraint=True, max_budget_inr=100000.0)

Expected result: `has_budget_constraint=True` and a numeric `max_budget_inr` value (around 100,000 for '1l').


Define `check_market_price`. It searches the web for current retail price/MRP of a product.


In [35]:
@tool
def check_market_price(product_name : str) -> str:
    """Searches the live web to find the current retail price or MRP for a specific product model"""

    search = DuckDuckGoSearchRun()
    try:
        return search.run(f"{product_name} current retail price MRP buy online")
    except:
        return f"Error fetching price : {str(e)}"

Define the `BudgetEvaluation` schema and build the price evaluator chain. It returns estimated price, whether it is within budget, and notes.


In [42]:
class BudgetEvaluation(BaseModel):
    estimated_market_price: float = Field(description="The numeric INR value for the product")
    is_within_budget: bool = Field(description = "True if estimated_market_price is less than or equal to the allowed budget")
    price_notes: str = Field(description= "A quick breakdown of pricing variants found")

pricing_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a pricing specialist. Look at the raw web search data, extract the numeric price of the product, and evaluate if it fits within the user's budget limit of INR {allowed_budget}"),
    ("human", "Evaluate pricing for the product. Web price data : {price_tool_context}")
    
])


price_evaluator_chain = pricing_agent_prompt | llm.with_structured_output(BudgetEvaluation)



Update `user_query` to a budget-aware query so the pipeline will trigger price checks.


In [38]:
user_query = "Top laptop brands within 1.2l"

## Section 8: Review Analysis & Final Recommendation

The last two agents collect user sentiment and produce a buying verdict.


Define `search_reviews`. It searches Reddit and TechRadar for user reviews, pros, and cons.


In [44]:
@tool
def search_reviews(product_name: str) -> str:
    """Searches the live web to find user reviews, pros, and cons for a specific product model."""
    search = DuckDuckGoSearchRun()
    try:
        return search.run(f"{product_name} user reviews pros cons site:reddit.com OR site:techradar.com")
    except Exception as e:
        return f"Error fetching reviews: {str(e)}"


Define the `PublicSentiment` schema and reviewer chain. The LLM extracts top pros/cons and overall sentiment.


In [45]:
class PublicSentiment(BaseModel):
    product_name: str
    pros: List[str] = Field(description="Top 3 pros mentioned by users.")
    cons: List[str] = Field(description="Top 3 cons or complaints mentioned by users.")
    overall_sentiment: str = Field(description="Is the reception Positive, Mixed, or Negative?")

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a consumer review analyst. Use the context to gather recent user feedback. Be objective and capture what real users are saying."),
    ("human", "Analyze reviews for: {product_name}. Web Data: {tool_context}")
])
structured_reviewer = review_prompt | llm.with_structured_output(PublicSentiment)


Define the `FinalVerdict` schema and advisor chain. This agent combines specs, reviews, and optional budget data into a recommendation.


In [46]:
class FinalVerdict(BaseModel):
    verdict: str = Field(description="Should the user buy this? (e.g., 'Highly Recommended', 'Over Budget - Avoid', 'Good Value Choice')")
    justification: str = Field(description="A clear explanation balancing specs, reviews, and budget constraints.")
    target_audience: str = Field(description="Who is this product best suited for?")

advisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite product advisor. Provide a definitive buying recommendation. If budget data is present, prioritize checking if the item is financially viable for the user."),
    ("human", "Technical Specs:\n{specs}\n\nUser Reviews:\n{reviews}\n\nOptional Budget/Price Check Data:\n{budget_data}")
])
structured_advisor = advisor_prompt | llm.with_structured_output(FinalVerdict)

## Section 9: End-to-End Recommendation Pipeline

This cell ties everything together: routing, discovery, specs, conditional price check, reviews, and final verdict.


Run the full pipeline on the current `user_query` (`'Top laptop brands within 1.2l'`). It prints a structured verdict for each discovered product.


In [48]:
routing_decision = router_chain.invoke({"user_query": user_query})

for product in discovered_data.products_name:
    print(f" -- processing : {product} --- ")

    print(f"Agent 1:Fetching technical specifications ... ")
    spec_raw_data = research_technical_specs.invoke({"product_name" : product})
    print(spec_raw_data)
    specs_output = structured_researcher.invoke({"product_to_research":product,"tool_context":spec_raw_data})
    print("-------specs Output -----")
    print(specs_output)

    budget_report_json = "{}"
    if routing_decision.has_budget_constraint:
        price_raw_data = check_market_price.invoke({"product_name": specs_output.product_name})
        budget_analysis = price_evaluator_chain.invoke({
        "allowed_budget" : routing_decision.max_budget_inr,
        "price_tool_context" : price_raw_data
        })

        print(f"     ↳ Est. Price: INR {budget_analysis.estimated_market_price} | Within Budget: {budget_analysis.is_within_budget}")
        budget_report_json = budget_analysis.model_dump_json()

    review_tool_result = search_reviews.invoke({"product_name": specs_output.product_name})

    reviews_output = structured_reviewer.invoke({
            "product_name": specs_output.product_name, 
            "tool_context": review_tool_result
        })

    final_recommendation = structured_advisor.invoke({
            "specs": specs_output.model_dump_json(),
            "reviews": reviews_output.model_dump_json(),
            "budget_data": budget_report_json
        })

        # --- Print Final Summary Output For This Product ---
    print(f"  📋 VERDICT FOR {specs_output.product_name.upper()}:")
    print(f"     Status:        {final_recommendation.verdict}")
    print(f"     Target User:   {final_recommendation.target_audience}")
    print(f"     Reasoning:     {final_recommendation.justification}")
    print("-" * 60 + "\n")


    

 -- processing : Apple MacBook Pro 16-inch (2023) --- 
Agent 1:Fetching technical specifications ... 
1 week ago - The 16-inch version is bundled with a 140 W GaN power supply that supports USB-C Power Delivery 3.1, though only MagSafe supports full-speed charging as the machine's USB-C ports are limited to 100 W. On January 17, 2023, Apple announced updated ... 1 week ago - The new models come with the Apple M2 Pro and M2 Max, can be configured with up to 96 GB of RAM (up from 64 GB), support HDMI 2.1 that can drive an 8K external display (the 2021 models supported HDMI 2.0), and support faster Wi-Fi 6E. April 7, 2026 - 16.2-inch (diagonal) Liquid Retina XDR display;2 3456-by-2234 native resolution at 254 pixels per inch January 2, 2026 - The MacBook Pro (16-inch, 2023) was introduced on 17 January 2023 (2023-01-17). It was made available for preorder on 17 January 2023 (2023-01-17), and for purchase... March 4, 2026 - Detailed technical specifications for the Apple MacBook Pro (16-in

Expected result: for each product you should see the model name, estimated price (if budget detected), whether it fits the budget, and a final verdict such as 'Highly Recommended' or 'Over Budget - Avoid'.


## End of Notebook

You have now walked through building a multi-agent product recommendation system from validation schemas to a final verdict pipeline.
